<a href="https://colab.research.google.com/github/julianzamb/5-DoF-in-Colab/blob/main/5DoF_3D_SIMULATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q pythreejs
from google.colab import output
output.enable_custom_widget_manager()

In [9]:
import math, time
import numpy as np
from scipy.interpolate import CubicSpline
from pythreejs import (
    Scene, PerspectiveCamera, Renderer, OrbitControls, Object3D, Mesh,
    BoxGeometry, CylinderGeometry, MeshStandardMaterial, AmbientLight,
    HemisphereLight, GridHelper, AxesHelper,
    AnimationMixer, AnimationClip, AnimationAction, QuaternionKeyframeTrack
)
from ipywidgets import (ToggleButton, Button, Textarea, FloatText, Checkbox,
                        HTML, Label, VBox, HBox, Layout)
from IPython.display import display

# ---------- Quaternion helpers (x, y, z, w) -- avoids Euler 'xyz' sync bug ----------
def q_x(a): return (math.sin(a/2), 0.0, 0.0, math.cos(a/2))
def q_y(a): return (0.0, math.sin(a/2), 0.0, math.cos(a/2))
def q_z(a): return (0.0, 0.0, math.sin(a/2), math.cos(a/2))

# ---------- Kinematic parameters (meters) -- UNCHANGED ----------
BASE_H = 0.10
L0, L1, L2, L3, L4 = 0.20, 0.60, 0.50, 0.30, 0.20
W = 0.08

def mat(color):
    return MeshStandardMaterial(color=color, metalness=0.2, roughness=0.6)

def link(length, color, width=W):
    """Box spanning local z = 0 -> length."""
    return Mesh(BoxGeometry(width=width, height=width, depth=length), mat(color),
                position=(0, 0, length / 2))

def joint_cyl(axis, color, radius=0.06, length=0.12):
    m = Mesh(CylinderGeometry(radiusTop=radius, radiusBottom=radius,
                              height=length, radialSegments=24), mat(color))
    if axis == 'z':                      # cylinders are Y-aligned; turn Y -> Z
        m.quaternion = q_x(math.pi / 2)
    return m

# ---------- Kinematic chain -- UNCHANGED ----------
base = Mesh(BoxGeometry(width=0.4, height=0.4, depth=BASE_H), mat('#444444'),
            position=(0, 0, BASE_H / 2))

j1 = Object3D(position=(0, 0, BASE_H))
j1.add(joint_cyl('z', '#cc3333'))
j1.add(link(L0, '#888888'))

j2 = Object3D(position=(0, 0, L0))
j2.add(joint_cyl('y', '#cc3333'))
j2.add(link(L1, '#3b82f6'))

j3 = Object3D(position=(0, 0, L1))
j3.add(joint_cyl('y', '#cc3333'))
j3.add(link(L2, '#22c55e'))

j4 = Object3D(position=(0, 0, L2))
j4.add(joint_cyl('y', '#cc3333'))
j4.add(link(L3, '#eab308'))

j5 = Object3D(position=(0, 0, L3))
j5.add(joint_cyl('z', '#cc3333', radius=0.05, length=0.06))
j5.add(link(L4, '#a855f7', width=0.05))

# ---------- Gripper -- UNCHANGED ----------
PLATE_T = 0.02
FINGER_W, FINGER_D, FINGER_H = 0.02, 0.10, 0.03
FINGER_Z = PLATE_T + FINGER_D / 2
X_CLOSED, X_OPEN = 0.0105, 0.06

gripper = Object3D(position=(0, 0, L4))

plate = Mesh(BoxGeometry(width=0.16, height=0.05, depth=PLATE_T), mat('#222222'),
             position=(0, 0, PLATE_T / 2))
finger_l = Mesh(BoxGeometry(width=FINGER_W, height=FINGER_H, depth=FINGER_D),
                mat('#f97316'), position=(-X_CLOSED, 0, FINGER_Z))
finger_r = Mesh(BoxGeometry(width=FINGER_W, height=FINGER_H, depth=FINGER_D),
                mat('#f97316'), position=(X_CLOSED, 0, FINGER_Z))

gripper.add(plate); gripper.add(finger_l); gripper.add(finger_r)
j5.add(gripper)

base.add(j1); j1.add(j2); j2.add(j3); j3.add(j4); j4.add(j5)

for _n, _j in zip(['j1', 'j2', 'j3', 'j4', 'j5'], [j1, j2, j3, j4, j5]):
    _j.name = _n
JOINT_AXES = ['z', 'y', 'y', 'y', 'z']

# ---------- Scene (Z-up) -- UNCHANGED ----------
grid = GridHelper(2, 20)
grid.quaternion = q_x(math.pi / 2)
axes = AxesHelper(0.5)

camera = PerspectiveCamera(position=(1.8, 1.8, 1.4), up=(0, 0, 1),
                           fov=50, aspect=800/600)

scene = Scene(children=[
    base, grid, axes, camera,
    AmbientLight(color='#ffffff', intensity=0.5),
    HemisphereLight(skyColor='#ffffff', groundColor='#555555',
                    intensity=0.9, position=(0, 0, 1)),
], background='#f0f0f0')

controls = OrbitControls(controlling=camera, target=(0, 0, 0.7))
renderer = Renderer(camera=camera, scene=scene, controls=[controls],
                    antialias=True, width=800, height=600)

# =====================================================================
# Joint pose application (same axes as the old slider callbacks:
# J1 = Z, J2/J3/J4 = Y, J5 = Z)
# =====================================================================
pose = {'q': np.zeros(5)}            # current joint angles in degrees

def apply_pose(q_deg, force=False):
    """Send only the joints that changed, rounded to cut float noise and message count."""
    q = np.round(np.asarray(q_deg, dtype=float), 3)
    prev = pose['q']
    a = np.radians(q)
    if force or q[0] != prev[0]: j1.quaternion = q_z(a[0])
    if force or q[1] != prev[1]: j2.quaternion = q_y(a[1])
    if force or q[2] != prev[2]: j3.quaternion = q_y(a[2])
    if force or q[3] != prev[3]: j4.quaternion = q_y(a[3])
    if force or q[4] != prev[4]: j5.quaternion = q_z(a[4])
    pose['q'] = q

# Same ranges the old sliders used (degrees)
JOINT_NAMES = ['J1', 'J2', 'J3', 'J4', 'J5']
LIMITS = np.array([[-180, 180], [-90, 90], [-150, 150], [-120, 120], [-180, 180]], float)

# =====================================================================
# Trajectory planner
# =====================================================================
class TrajectoryPlanner:
    """Joint-space cubic-spline trajectory through N x 5 waypoints (degrees).

    Waypoints are spaced uniformly in time over [0, T]. Clamped boundary
    conditions give zero velocity at the start and end, so the arm
    starts and stops smoothly.
    """
    def __init__(self, waypoints, T):
        wp = np.asarray(waypoints, dtype=float)
        if wp.ndim != 2 or wp.shape[1] != 5:
            raise ValueError(f"Waypoints must be N x 5, got shape {wp.shape}")
        if wp.shape[0] < 2:
            raise ValueError("Need at least 2 waypoints")
        if T <= 0:
            raise ValueError("T must be > 0")
        self.T = float(T)
        self.waypoints = wp
        self.times = np.linspace(0.0, self.T, wp.shape[0])
        self.spline = CubicSpline(self.times, wp, axis=0, bc_type='clamped')

    def __call__(self, t):
        """Joint angles (deg) at time t, clamped to [0, T]."""
        return self.spline(np.clip(t, 0.0, self.T))

    def velocity(self, t):
        return self.spline(np.clip(t, 0.0, self.T), 1)

    def sample(self, fps=30):
        ts = np.linspace(0.0, self.T, max(2, int(round(self.T * fps)) + 1))
        return ts, self.spline(ts)

def parse_waypoints(text):
    """One row per line, 5 comma-separated angles in degrees. '#' starts a comment."""
    rows = []
    for i, line in enumerate(text.splitlines(), 1):
        line = line.split('#')[0].strip()
        if not line:
            continue
        try:
            vals = [float(p) for p in line.split(',')]
        except ValueError:
            raise ValueError(f"Line {i}: non-numeric value")
        if len(vals) != 5:
            raise ValueError(f"Line {i}: expected 5 values, got {len(vals)}")
        rows.append(vals)
    return np.array(rows, dtype=float).reshape(-1, 5)

def check_limits(q):
    """Return a message if any sample in q (M x 5) leaves the joint limits, else None."""
    for j in range(5):
        lo, hi = LIMITS[j]
        if q[:, j].min() < lo - 1e-6 or q[:, j].max() > hi + 1e-6:
            return (f"{JOINT_NAMES[j]} leaves its range [{lo:.0f}, {hi:.0f}] deg "
                    f"(reaches {q[:, j].min():.1f} .. {q[:, j].max():.1f})")
    return None

# =====================================================================
# UI
# =====================================================================
FPS = 30

DEFAULT_WAYPOINTS = """# J1, J2, J3, J4, J5  (degrees)
0, 0, 0, 0, 0
45, 30, -60, 40, 90
-45, 60, -90, -30, -90
0, 0, 0, 0, 0"""

wp_text = Textarea(value=DEFAULT_WAYPOINTS, rows=8,
                   placeholder='J1, J2, J3, J4, J5  (one row per waypoint, degrees)',
                   layout=Layout(width='400px'))
t_input = FloatText(value=6.0, description='T (s)', step=0.5,
                    layout=Layout(width='200px'))
from_current = Checkbox(value=True, description='Start from current pose',
                        indent=False, layout=Layout(width='400px'))
run_btn = Button(description='Run Trajectory', button_style='success',
                 icon='play', layout=Layout(width='400px'))
status = HTML(value='Ready.')

def set_status(msg, error=False):
    color = '#b91c1c' if error else '#374151'
    status.value = f'<span style="color:{color}">{msg}</span>'

anim = {'action': None}
QFUN = {'x': q_x, 'y': q_y, 'z': q_z}
CLIP_FPS = 60                                  # keyframe density; playback runs at display rate

def build_clip(planner):
    """Sample the spline once and pack it into one quaternion track per joint."""
    ts, q = planner.sample(CLIP_FPS)
    tracks = []
    for j in range(5):
        f = QFUN[JOINT_AXES[j]]
        vals = []
        for deg in q[:, j]:
            vals.extend(f(math.radians(deg)))
        tracks.append(QuaternionKeyframeTrack(
            name=f'j{j+1}.quaternion', times=ts.tolist(), values=vals))
    return AnimationClip(tracks=tracks, duration=float(planner.T))

def run_trajectory(_):
    run_btn.disabled = True
    try:
        wp = parse_waypoints(wp_text.value)
        if from_current.value and (wp.shape[0] == 0 or
                                   not np.allclose(wp[0], pose['q'], atol=1e-3)):
            wp = np.vstack([pose['q'], wp])
        planner = TrajectoryPlanner(wp, t_input.value)

        _, q_samples = planner.sample(FPS)
        msg = check_limits(q_samples)
        if msg:
            set_status(msg, error=True)
            return

        if anim['action'] is not None:         # stop the previous run
            anim['action'].stop()

        clip = build_clip(planner)
        mixer = AnimationMixer(base)           # root = base, so 'j1'..'j5' resolve
        action = AnimationAction(mixer, clip, base,
                                 loop='LoopOnce', repetitions=1,
                                 clampWhenFinished=True)
        anim['action'] = action

        set_status(f'Running: {wp.shape[0]} waypoints, T = {planner.T:.2f} s ...')
        action.play()                          # one message; the browser does the rest

        pose['q'] = np.round(wp[-1], 3)        # Python-side record of the end pose
        time.sleep(planner.T + 0.1)            # only to re-enable the button; sends nothing
        set_status('Done.')
    except Exception as e:
        set_status(f'Error: {e}', error=True)
    finally:
        run_btn.disabled = False

run_btn.on_click(run_trajectory)

# ---------- Gripper toggle + animation -- UNCHANGED ----------
grip = {'t': 0.0}

def set_gripper(t):
    x = X_CLOSED + (X_OPEN - X_CLOSED) * t
    finger_l.position = (-x, 0, FINGER_Z)
    finger_r.position = (x, 0, FINGER_Z)

def animate_gripper(target, duration=0.35, steps=14):
    start = grip['t']
    for i in range(1, steps + 1):
        s = i / steps
        s = s * s * (3 - 2 * s)
        grip['t'] = start + (target - start) * s
        set_gripper(grip['t'])
        time.sleep(duration / steps)
    grip['t'] = target
    set_gripper(target)
    time.sleep(0.05)
    set_gripper(target)

gripper_toggle = ToggleButton(value=False, description='Open/Close Gripper',
                              button_style='info', icon='hand-paper',
                              tooltip='Off = closed, On = open',
                              layout=Layout(width='400px'))

def on_toggle(change):
    animate_gripper(1.0 if change['new'] else 0.0)

gripper_toggle.observe(on_toggle, names='value')

set_gripper(0.0)
apply_pose(np.zeros(5))

panel = VBox([
    Label('Waypoints (deg): J1, J2, J3, J4, J5 per row'),
    wp_text, t_input, from_current, run_btn, status, gripper_toggle,
])
display(HBox([renderer, panel]))